# V1 Demo

Spec Step 10 deliverable: a Jupyter notebook that loads the V1 exports and demonstrates basic visualization for each criterion.

**What you can do here:**
- Load any V1 Tier-1 criterion as a GeoDataFrame.
- Inspect the per-layer metadata (source, vintage, license, native unit, state + trajectory framing, sovereignty axes).
- Render a quick map for each.

**What's NOT shown here (deliberately):**
- The four Tier-2 criteria (climate, soil_carbon, solar_pv, population). These exist in V1 only as per-region curated dossier values, not ingested layers — they are not loadable through `v1_loader.load()` and do NOT count toward the V1 ship gate. See `Docs/coverage-report.md` for the honest Tier-1/Tier-2 distinction.
- `forest_change` is served live as XYZ tiles (Hansen/GFW); see `data/v1-exports/forest_change.README.md`.

In [ ]:
import sys
sys.path.insert(0, '../scripts')
from v1_loader import load, criteria, load_metadata
print('V1 criteria available:', criteria)

## legal_ownership — the r4-promoted gate criterion (Askja #10)
Per-jurisdiction structured layer, 20/20 in-scope regions, carries red-line collective-ownership-legality data.

In [ ]:
legal = load('legal_ownership')
print(f'{len(legal)} regions, geometry: {legal.geometry.geom_type.iloc[0]}')
print()
print('Metadata sidecar:')
for k, v in legal.attrs.items():
    if k != 'properties_schema':
        print(f'  {k}: {v}')
print()
print('Sample fields per region:')
legal[['region_id', 'multi_household_residence_as_of_right', 'regulatory_direction', 'key_legal_restriction']].head(10)

In [ ]:
# Filter example: regions where multi-household residence is allowed as-of-right.
as_of_right = legal[legal['multi_household_residence_as_of_right'] == 'yes']
print(f'{len(as_of_right)} of {len(legal)} regions allow multi-household residence as-of-right:')
as_of_right[['region_id', 'key_legal_restriction']]

## water_stress — WRI Aqueduct 4.0 (2050 BAU projection)

In [ ]:
ws = load('water_stress')
print(f'{len(ws)} basins ({ws.geometry.geom_type.iloc[0]}); continents: {ws["continent"].value_counts().to_dict()}')
ws.plot(column='score', cmap='YlOrRd', legend=True, figsize=(12, 6))

## conflict — UCDP GED v25.1, observed events 2015–2024

In [ ]:
conflict = load('conflict')
print(f'{len(conflict)} events; continents: {conflict["continent"].value_counts().to_dict()}')
conflict.plot(markersize=0.5, alpha=0.4, figsize=(12, 6))

## regen_network — OSM/GEN ecovillage and intentional-community points
Note: coverage is patchy by design (volunteer-tagged OSM undercounts the full GEN directory, see metadata).

In [ ]:
regen = load('regen_network')
print(f'{len(regen)} sites; continents: {regen["continent"].value_counts().to_dict()}')
regen.plot(markersize=8, color='green', figsize=(12, 6))

## QGIS exports
Every loaded layer above is also available as a GeoPackage in `data/v1-exports/<criterion>.gpkg` — drag into QGIS as a vector layer.
`forest_change` is XYZ-tile-served; see the README in the exports folder for the QGIS XYZ URL.